# BC Ablation Analysis: Metrics + Feature Attribution (Daily LSTM)

Compare daily LSTM grid search results with and without boundary condition (BC) inputs across Russian River basins, then use Integrated Gradients (IG) to explain what each model attends to and how attribution shifts under BC removal.

- **Guerneville**: 2 BCs removed (UKIAH CA FLOW, GEYSERVILLE CA FLOW)
- **Hopland**: 1 BC removed (UKIAH CA FLOW)
- **Calpella**: 1 BC removed (POTTER VALLEY CA FLOW)
- **Warm Springs**: 0 BCs (reference - no ablation needed)

BC removal applies to both LSTM (no-physics) and PILSTM (physics) variants.

**Experiments:**
- **BASELINE** - all features including observed upstream flow BCs
- **NOBC** - observed flow BCs removed
- **NOBC_V2** - observed + HMS-derived flows removed (zero flow information)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, json, yaml, sys
from pathlib import Path
from matplotlib.patches import Patch
import warnings; warnings.simplefilter('ignore', FutureWarning)

sys.path.insert(0, str(Path('../..').resolve()))
from UCB_training.ucb_captum import (
    run_ig_analysis, save_attributions, load_attributions,
    rank_features, build_combined_df, plot_ig_side_by_side,
    plot_ig_rank_shift, plot_ig_heatmap,
)
print('imports OK')

In [ ]:
OUTPUTS = Path('../../outputs')
DATA_DIR = Path('../../russian_river_data')
RUNS_BASE = OUTPUTS
MODEL_TYPE = 'daily'

FIG_DIR = OUTPUTS / '_all_basins' / 'figs_bc_ablation'
METRICS_DIR = FIG_DIR / 'metrics_baseline_vs_nobc'
METRICS_3T_DIR = FIG_DIR / 'metrics_3tier'
IG_CACHE_DIR = OUTPUTS / '_all_basins' / 'ig_analysis' / 'cache'
IG_CSV_PATH = OUTPUTS / '_all_basins' / 'ig_analysis' / 'ig_all_daily_runs.csv'
for d in [FIG_DIR, METRICS_DIR, METRICS_3T_DIR, IG_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

basins_with_bc = ['guerneville', 'hopland', 'calpella']
all_basins = basins_with_bc + ['warm_springs']

run_stamps = {
    ('guerneville', 'BASELINE'): 'BASELINE_20250815T000000Z',
    ('guerneville', 'BASELINE_NOBC'): 'BASELINE_NOBC_20260219T195832Z',
    ('guerneville', 'BASELINE_NOBC_V2'): 'BASELINE_NOBC_V2_20260220T071721Z',
    ('hopland', 'BASELINE'): 'BASELINE_20250815T000000Z',
    ('hopland', 'BASELINE_NOBC'): 'BASELINE_NOBC_20260219T232252Z',
    ('hopland', 'BASELINE_NOBC_V2'): 'BASELINE_NOBC_V2_20260220T075114Z',
    ('calpella', 'BASELINE'): 'BASELINE_20250815T000000Z',
    ('calpella', 'BASELINE_NOBC'): 'BASELINE_NOBC_20260220T030916Z',
    ('calpella', 'BASELINE_NOBC_V2'): 'BASELINE_NOBC_V2_20260220T082247Z',
}

IG_RUN_STAMPS = {
    ('guerneville', 'BASELINE'): '20250815T000000Z',
    ('hopland', 'BASELINE'): '20250815T000000Z',
    ('calpella', 'BASELINE'): '20250815T000000Z',
    ('warm_springs', 'BASELINE'): '20250815T000000Z',
    ('guerneville', 'BASELINE_NOBC'): '20260219T195832Z',
    ('hopland', 'BASELINE_NOBC'): '20260219T232252Z',
    ('calpella', 'BASELINE_NOBC'): '20260220T030916Z',
    ('guerneville', 'BASELINE_NOBC_V2'): '20260220T071721Z',
    ('hopland', 'BASELINE_NOBC_V2'): '20260220T075114Z',
    ('calpella', 'BASELINE_NOBC_V2'): '20260220T082247Z',
}

BC_FEATURES = {
    'guerneville': ['UKIAH CA FLOW USGS-MERGED', 'GEYSERVILLE CA FLOW USGS-MERGED'],
    'hopland': ['UKIAH CA FLOW USGS-MERGED'],
    'calpella': ['POTTER VALLEY CA FLOW USGS_ADJUSTED'],
}

LABELS = ['BASELINE', 'BASELINE_NOBC', 'BASELINE_NOBC_V2']
LABEL_SHORT = {'BASELINE': 'BC', 'BASELINE_NOBC': 'NOBC', 'BASELINE_NOBC_V2': 'V2'}
DISPLAY_METRICS = ['NSE', 'KGE', 'Pearson-r', 'RMSE', 'FHV', 'Peak-Timing', 'Peak-MAPE']
HIGHER_BETTER = {'NSE', 'KGE', 'Pearson-r'}
LOWER_BETTER = {'RMSE', 'Peak-Timing', 'Peak-MAPE'}
CLOSER_ZERO = {'FHV'}
PER_BASIN = {'RMSE'}
METRIC_RANGES = {'NSE': (0.6, 1.0), 'KGE': (0.6, 1.0), 'Pearson-r': (0.8, 1.0), 'FHV': (0, 25), 'Peak-Timing': (0, 1.0), 'Peak-MAPE': (0, 50)}

def load_grid(basin, label, variant):
    fname = f'{basin}_{MODEL_TYPE}_{label}_{variant}_gridsearch.csv'
    path = OUTPUTS / basin / f'{MODEL_TYPE}_shared' / 'hyperparams' / fname
    if not path.exists(): return None
    df = pd.read_csv(path); df['basin'], df['label'], df['variant'] = basin, label, variant
    return df

def load_best(basin, label):
    fname = f'{basin}_{MODEL_TYPE}_{label}_hyperparams.csv'
    path = OUTPUTS / basin / f'{MODEL_TYPE}_shared' / 'hyperparams' / fname
    if not path.exists(): return None
    df = pd.read_csv(path); df['basin'], df['label'] = basin, label
    return df

def load_metrics(basin, label, period):
    stamp = run_stamps.get((basin, label))
    if stamp is None: return None
    tag = 'val' if period == 'validation' else 'test'
    path = OUTPUTS / basin / 'daily' / stamp / 'metrics' / period / f'{basin}_daily_{tag}_metrics.csv'
    if not path.exists(): return None
    return pd.read_csv(path, index_col=0)

def build_heatmap_df(basin, period_dir):
    cols = {}
    for label, tag in [('BASELINE', 'BC'), ('BASELINE_NOBC', 'NOBC')]:
        df = load_metrics(basin, label, period_dir)
        if df is None: continue
        for model in ['LSTM', 'PILSTM']:
            cols[f'{model}\n{tag}'] = {m: float(df.loc[m, model]) for m in DISPLAY_METRICS}
    return pd.DataFrame(cols, index=DISPLAY_METRICS)

def build_heatmap_3t(basin, period_dir):
    cols = {}
    for label in LABELS:
        tag = LABEL_SHORT[label]
        df = load_metrics(basin, label, period_dir)
        if df is None: continue
        for model in ['LSTM', 'PILSTM']:
            cols[f'{model}\n{tag}'] = {m: float(df.loc[m, model]) for m in DISPLAY_METRICS}
    return pd.DataFrame(cols, index=DISPLAY_METRICS)

def norm_for_color(hm_df):
    norm = hm_df.copy()
    for m in DISPLAY_METRICS:
        row = hm_df.loc[m]
        if m in PER_BASIN:
            rng = row.max() - row.min() + 1e-10
            norm.loc[m] = ((row.max() - row) / rng).clip(0, 1)
        elif m in HIGHER_BETTER:
            lo, hi = METRIC_RANGES[m]; norm.loc[m] = ((row - lo) / (hi - lo)).clip(0, 1)
        elif m in CLOSER_ZERO:
            lo, hi = METRIC_RANGES[m]; norm.loc[m] = ((hi - row.abs()) / (hi - lo)).clip(0, 1)
        else:
            lo, hi = METRIC_RANGES[m]; norm.loc[m] = ((hi - row) / (hi - lo)).clip(0, 1)
    return norm

daily_best_df = pd.concat([d for b in all_basins for l in ['BASELINE', 'BASELINE_NOBC'] if (d := load_best(b, l)) is not None], ignore_index=True)
daily_grid_df = pd.concat([d for b in basins_with_bc for l in ['BASELINE', 'BASELINE_NOBC'] for v in ['no_physics', 'physics'] if (d := load_grid(b, l, v)) is not None], ignore_index=True)
print(f'Daily best-param rows: {len(daily_best_df)}  |  Daily grid search rows: {len(daily_grid_df)}')

In [ ]:
def _find_stored_runs_json(basin, experiment):
    path = RUNS_BASE / basin / 'daily_shared' / 'runs' / f'{basin}_daily_{experiment}_stored_runs.json'
    if path.exists(): return json.loads(path.read_text())
    return None

def _scan_testing_runs(base_dir):
    found = {}
    for d in sorted(base_dir.glob('testing_run_*')):
        cfg_path = d / 'config.yml'
        if not cfg_path.exists(): continue
        cfg = yaml.safe_load(cfg_path.read_text())
        pt = 'PHYS' if cfg.get('physics_informed', False) else 'NP'
        found[pt] = d
    return found

def discover_runs():
    runs = {}
    for (basin, experiment), stamp in IG_RUN_STAMPS.items():
        base_dir = RUNS_BASE / basin / 'daily_shared' / 'runs' / f'{experiment}_{stamp}'
        if not base_dir.exists():
            print(f'  SKIP {basin}/{experiment}: dir not found'); continue
        resolved = False
        stored = _find_stored_runs_json(basin, experiment)
        if stored:
            for phys_type, key_prefs in [('NP', ['no_physics_test', 'no_physics']), ('PHYS', ['physics_test', 'physics'])]:
                for key in key_prefs:
                    if key in stored:
                        run_dir = base_dir / stored[key][0]
                        if run_dir.exists():
                            runs[(basin, experiment, phys_type)] = run_dir
                            resolved = True; break
        if not resolved:
            scanned = _scan_testing_runs(base_dir)
            for pt, d in scanned.items(): runs[(basin, experiment, pt)] = d
            if scanned: resolved = True
        if not resolved:
            print(f'  SKIP {basin}/{experiment}: no valid testing_run dirs found')
    return runs

def compute_or_load_all():
    RUNS = discover_runs()
    expected_keys = set(RUNS.keys())
    if IG_CSV_PATH.exists():
        existing = pd.read_csv(IG_CSV_PATH)
        csv_keys = set(existing[['basin', 'experiment', 'phys_type']].drop_duplicates().itertuples(index=False, name=None))
        if csv_keys == expected_keys:
            print(f'CSV cache hit: {len(csv_keys)} runs, {len(existing)} rows')
            return existing
        print(f'CSV stale ({len(csv_keys)} cached vs {len(expected_keys)} expected)')
    results = {}
    for (basin, experiment, phys_type), run_dir in sorted(RUNS.items()):
        pt_path = IG_CACHE_DIR / f'{basin}_{experiment}_{phys_type}.pt'
        if pt_path.exists():
            attrs, meta = load_attributions(pt_path)
            df = rank_features(attrs, meta['feature_names'])
            print(f'.pt hit:  {basin}/{experiment}/{phys_type}')
        else:
            print(f'compute:  {basin}/{experiment}/{phys_type} ...', end=' ')
            df, attrs, feat_names = run_ig_analysis(run_dir, period='test', n_samples=50, data_dir=str(DATA_DIR), return_attributions=True)
            save_attributions(attrs, feat_names, pt_path, basin=basin, experiment=experiment, phys_type=phys_type)
            print('done')
        results[(basin, experiment, phys_type)] = df
    combined = build_combined_df(results)
    combined.to_csv(IG_CSV_PATH, index=False)
    print(f'Saved: {IG_CSV_PATH.name} - {len(combined)} rows, {len(results)} runs')
    return combined

all_ig = compute_or_load_all()
baseline_ig = all_ig[all_ig['experiment'] == 'BASELINE']

---

## Q1: What Do Models Attend To? BASELINE Feature Attribution - LSTM vs PILSTM

Before examining how BC removal shifts attribution, we first establish what each model attends to under the full-feature BASELINE. Each panel shows the top-15 features by mean absolute IG attribution over the test period.

In [ ]:
Q1_DIR = FIG_DIR / 'ig_q1_baseline_np_vs_phys'
Q1_DIR.mkdir(parents=True, exist_ok=True)

def compare_lstm_pilstm(basin):
    np_df = baseline_ig[(baseline_ig['basin'] == basin) & (baseline_ig['phys_type'] == 'NP')]
    phys_df = baseline_ig[(baseline_ig['basin'] == basin) & (baseline_ig['phys_type'] == 'PHYS')]
    title = f'{basin.replace("_", " ").title()} - BASELINE: LSTM vs PILSTM'
    fig, axes = plot_ig_side_by_side([np_df, phys_df], ['LSTM', 'PILSTM'], title=title)
    fig.savefig(Q1_DIR / f'{basin}_baseline_lstm_vs_pilstm.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
compare_lstm_pilstm('guerneville')

In [ ]:
compare_lstm_pilstm('hopland')

In [ ]:
compare_lstm_pilstm('calpella')

In [ ]:
compare_lstm_pilstm('warm_springs')

---

## Metrics: BASELINE vs NOBC - Performance Impact of BC Removal

How does removing observed upstream flow BCs affect model performance? NSE comparison, grid search distributions, validation + test bars, and full multi-metric heatmaps.

In [ ]:
daily_nse_summary = daily_best_df.pivot_table(index=['basin', 'model_type'], columns='label', values='NSE', aggfunc='first')
if {'BASELINE', 'BASELINE_NOBC'}.issubset(daily_nse_summary.columns):
    daily_nse_summary['delta_NSE'] = daily_nse_summary['BASELINE_NOBC'] - daily_nse_summary['BASELINE']

hp_cols = [c for c in ['hidden_size', 'epochs', 'seq_length'] if c in daily_best_df.columns]
daily_hp_pivot = daily_best_df.pivot_table(index=['basin', 'model_type'], columns='label', values=hp_cols, aggfunc='first')

print('Daily NSE comparison (validation):'); display(daily_nse_summary.round(4))
print('\nDaily best hyperparams:'); display(daily_hp_pivot)

In [ ]:
s = daily_nse_summary['delta_NSE'].dropna()
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2196F3' if v >= 0 else '#F44336' for v in s.values]
ax.barh(range(len(s)), s.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(s)))
ax.set_yticklabels([f'{b} ({m})' for b, m in s.index], fontsize=11)
ax.set_xlabel('$\\Delta$ NSE (NOBC $-$ BASELINE)', fontsize=12)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Daily LSTM: Impact of removing boundary conditions', fontsize=13)
for i, v in enumerate(s.values):
    ax.text(v + 0.002 * np.sign(v), i, f'{v:+.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(METRICS_DIR / 'daily_delta_nse.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
BC_COLOR, NOBC_COLOR = '#00E676', '#FF6D00'
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for i, basin in enumerate(basins_with_bc):
    ax = axes[i]; sub = daily_grid_df[daily_grid_df['basin'] == basin]
    positions, labels, data, colors = [], [], [], []
    pos = 0
    for var_label, var_name in [('no_physics', 'LSTM'), ('physics', 'PILSTM')]:
        for run_label, color in [('BASELINE', BC_COLOR), ('BASELINE_NOBC', NOBC_COLOR)]:
            vals = sub[(sub['variant'] == var_label) & (sub['label'] == run_label)]['NSE'].values
            if len(vals) > 0:
                data.append(vals); positions.append(pos)
                labels.append(f'{var_name}\n{"BC" if run_label == "BASELINE" else "NOBC"}'); colors.append(color)
            pos += 1
        pos += 0.5
    bp = ax.boxplot(data, positions=positions, widths=0.6, patch_artist=True, showmeans=True, meanprops=dict(marker='D', markerfacecolor='white', markersize=5))
    for patch, c in zip(bp['boxes'], colors): patch.set_facecolor(c); patch.set_alpha(0.85); patch.set_edgecolor('black')
    ax.set_xticks(positions); ax.set_xticklabels(labels, fontsize=9)
    ax.set_title(basin.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    if i == 0: ax.set_ylabel('NSE (validation)', fontsize=12)
axes[-1].legend(handles=[Patch(facecolor=BC_COLOR, edgecolor='black', label='BASELINE (with BC)'), Patch(facecolor=NOBC_COLOR, edgecolor='black', label='NOBC (no BC)')], loc='lower right', fontsize=10)
fig.suptitle('Daily grid search NSE: BC vs No-BC', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(METRICS_DIR / 'daily_grid_boxplots.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
rows_vt = []
for basin in basins_with_bc:
    for label in ['BASELINE', 'BASELINE_NOBC']:
        for period in ['validation', 'test']:
            tag = 'val' if period == 'validation' else 'test'
            df = load_metrics(basin, label, period)
            if df is None: continue
            for model in ['LSTM', 'PILSTM']:
                rows_vt.append({'basin': basin, 'BC': 'BC' if label == 'BASELINE' else 'NOBC', 'period': tag, 'model': model, 'NSE': float(df.loc['NSE', model])})

full_df = pd.DataFrame(rows_vt)
pivot = full_df.pivot_table(index=['basin', 'model'], columns=['BC', 'period'], values='NSE')
pivot['delta_val'] = pivot[('NOBC', 'val')] - pivot[('BC', 'val')]
pivot['delta_test'] = pivot[('NOBC', 'test')] - pivot[('BC', 'test')]
display(pivot.round(4))

C = {'BC_val': '#00E676', 'NOBC_val': '#FF6D00', 'BC_test': '#2979FF', 'NOBC_test': '#FF1744'}
groups = [('BC', 'val', C['BC_val'], 'BC val'), ('NOBC', 'val', C['NOBC_val'], 'NOBC val'), ('BC', 'test', C['BC_test'], 'BC test'), ('NOBC', 'test', C['NOBC_test'], 'NOBC test')]
bar_w = 0.18

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for i, basin in enumerate(basins_with_bc):
    ax = axes[i]; sub = full_df[full_df['basin'] == basin]
    models = ['LSTM', 'PILSTM']
    x = np.arange(len(models))
    for j, (bc, period, color, key) in enumerate(groups):
        vals = [sub[(sub['model'] == m) & (sub['BC'] == bc) & (sub['period'] == period)]['NSE'].values for m in models]
        vals = [v[0] if len(v) > 0 else 0 for v in vals]
        bars = ax.bar(x + (j - 1.5) * bar_w, vals, bar_w, color=color, edgecolor='black', linewidth=0.5, label=key if i == 0 else '')
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, v + 0.003, f'{v:.3f}', ha='center', va='bottom', fontsize=7, rotation=90)
    ax.set_xticks(x); ax.set_xticklabels(models, fontsize=12, fontweight='bold')
    ax.set_title(basin.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    if i == 0: ax.set_ylabel('NSE', fontsize=12)
    ax.set_ylim(0.7, 1.0)
axes[0].legend(fontsize=9, loc='lower left')
fig.suptitle('Daily: Validation + Test NSE - BC vs No-BC', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(METRICS_DIR / 'daily_valtest_4color.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, basin in enumerate(basins_with_bc):
    ax = axes[i]; hm_df = build_heatmap_df(basin, 'validation'); norm_df = norm_for_color(hm_df)
    annot = hm_df.map(lambda v: f'{v:.1f}' if abs(v) >= 10 else f'{v:.3f}')
    sns.heatmap(norm_df, annot=annot, fmt='', cmap='RdYlGn', ax=ax, linewidths=0.5, cbar=False, vmin=0, vmax=1, annot_kws={'fontsize': 9})
    ax.set_title(basin.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    if i > 0: ax.set_ylabel('')
fig.suptitle('Daily: Validation Metrics - BC vs No-BC', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(METRICS_DIR / 'daily_heatmap_validation.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, basin in enumerate(basins_with_bc):
    ax = axes[i]; hm_df = build_heatmap_df(basin, 'test'); norm_df = norm_for_color(hm_df)
    annot = hm_df.map(lambda v: f'{v:.1f}' if abs(v) >= 10 else f'{v:.3f}')
    sns.heatmap(norm_df, annot=annot, fmt='', cmap='RdYlGn', ax=ax, linewidths=0.5, cbar=False, vmin=0, vmax=1, annot_kws={'fontsize': 9})
    ax.set_title(basin.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    if i > 0: ax.set_ylabel('')
fig.suptitle('Daily: Test Metrics - BC vs No-BC', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(METRICS_DIR / 'daily_heatmap_test.png', dpi=200, bbox_inches='tight')
plt.show()

---

## Q2: Where Does Attribution Shift? BASELINE vs NOBC

How does removing observed flow BCs change what the model attends to? Side-by-side IG attribution comparison for each BC basin, separately for LSTM and PILSTM. Features that absorb the removed BC's role should gain attribution.

**Removed features per basin:**
- **Guerneville** (2 BCs): UKIAH CA FLOW USGS-MERGED, GEYSERVILLE CA FLOW USGS-MERGED
- **Hopland** (1 BC): UKIAH CA FLOW USGS-MERGED
- **Calpella** (1 BC): POTTER VALLEY CA FLOW USGS_ADJUSTED

In [ ]:
Q2_DIR = FIG_DIR / 'ig_q2_baseline_vs_nobc'
Q2_DIR.mkdir(parents=True, exist_ok=True)

def compare_baseline_nobc(basin, phys_type):
    pt_label = 'LSTM' if phys_type == 'NP' else 'PILSTM'
    base_df = all_ig[(all_ig['basin'] == basin) & (all_ig['experiment'] == 'BASELINE') & (all_ig['phys_type'] == phys_type)]
    nobc_df = all_ig[(all_ig['basin'] == basin) & (all_ig['experiment'] == 'BASELINE_NOBC') & (all_ig['phys_type'] == phys_type)]
    removed = ', '.join(BC_FEATURES[basin])
    title = f'{basin.replace("_", " ").title()} ({pt_label}) - BASELINE vs NOBC\nRemoved: {removed}'
    fig, axes = plot_ig_side_by_side([base_df, nobc_df], ['BASELINE', 'NOBC'], title=title)
    fig.savefig(Q2_DIR / f'{basin}_{phys_type}_baseline_vs_nobc.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
compare_baseline_nobc('guerneville', 'NP')

In [ ]:
compare_baseline_nobc('guerneville', 'PHYS')

In [ ]:
compare_baseline_nobc('hopland', 'NP')

In [ ]:
compare_baseline_nobc('hopland', 'PHYS')

In [ ]:
compare_baseline_nobc('calpella', 'NP')

In [ ]:
compare_baseline_nobc('calpella', 'PHYS')

---

## 3-Tier Metrics: BASELINE vs NOBC vs NOBC_V2

NOBC_V2 removes all flow information (observed + HMS-derived), leaving models with only meteorological forcings. This is the strictest test of whether models can infer flow from weather alone.

In [ ]:
rows_3t = []
for basin in basins_with_bc:
    for label in LABELS:
        for period in ['validation', 'test']:
            tag = 'val' if period == 'validation' else 'test'
            df = load_metrics(basin, label, period)
            if df is None: continue
            for model in ['LSTM', 'PILSTM']:
                for m in DISPLAY_METRICS:
                    val = float(df.loc[m, model]) if m in df.index else np.nan
                    rows_3t.append({'basin': basin, 'label': LABEL_SHORT[label], 'period': tag, 'model': model, 'metric': m, 'value': val})

df_3t = pd.DataFrame(rows_3t)

nse_val = df_3t[(df_3t['metric'] == 'NSE') & (df_3t['period'] == 'val')].pivot_table(index=['basin', 'model'], columns='label', values='value')
nse_val = nse_val[['BC', 'NOBC', 'V2']]
nse_val['NOBC-BC'] = nse_val['NOBC'] - nse_val['BC']
nse_val['V2-BC'] = nse_val['V2'] - nse_val['BC']
nse_val['V2-NOBC'] = nse_val['V2'] - nse_val['NOBC']
display(nse_val.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(26, 6))
for i, basin in enumerate(basins_with_bc):
    hm = build_heatmap_3t(basin, 'validation')
    cn = norm_for_color(hm)
    fmt_vals = hm.map(lambda x: f'{x:.3f}' if abs(x) < 10 else f'{x:.1f}')
    sns.heatmap(cn, annot=fmt_vals, fmt='', cmap='RdYlGn', vmin=0, vmax=1, cbar=False, ax=axes[i], linewidths=0.5)
    axes[i].set_title(f'{basin.title()} - Validation', fontsize=12, fontweight='bold')
    axes[i].set_ylabel(''); axes[i].set_xlabel('')
fig.suptitle('3-Tier BC Ablation: Validation Heatmaps (Daily)', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(METRICS_3T_DIR / '3tier_validation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(26, 6))
for i, basin in enumerate(basins_with_bc):
    hm = build_heatmap_3t(basin, 'test')
    cn = norm_for_color(hm)
    fmt_vals = hm.map(lambda x: f'{x:.3f}' if abs(x) < 10 else f'{x:.1f}')
    sns.heatmap(cn, annot=fmt_vals, fmt='', cmap='RdYlGn', vmin=0, vmax=1, cbar=False, ax=axes[i], linewidths=0.5)
    axes[i].set_title(f'{basin.title()} - Test', fontsize=12, fontweight='bold')
    axes[i].set_ylabel(''); axes[i].set_xlabel('')
fig.suptitle('3-Tier BC Ablation: Test Heatmaps (Daily)', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(METRICS_3T_DIR / '3tier_test_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Q3: Zero Flow Attribution - NOBC vs NOBC_V2

NOBC_V2 removes all remaining flow-derived features (HMS simulated flows) beyond the observed BCs already removed in NOBC. This is the strictest ablation - the model has only meteorological forcings. How does attribution redistribute when the last flow signals are removed?

In [ ]:
Q3_DIR = FIG_DIR / 'ig_q3_nobc_vs_nobcv2'
Q3_DIR.mkdir(parents=True, exist_ok=True)

def compare_nobc_nobcv2(basin, phys_type):
    pt_label = 'LSTM' if phys_type == 'NP' else 'PILSTM'
    nobc_df = all_ig[(all_ig['basin'] == basin) & (all_ig['experiment'] == 'BASELINE_NOBC') & (all_ig['phys_type'] == phys_type)]
    v2_df = all_ig[(all_ig['basin'] == basin) & (all_ig['experiment'] == 'BASELINE_NOBC_V2') & (all_ig['phys_type'] == phys_type)]
    title = f'{basin.replace("_", " ").title()} ({pt_label}) - NOBC vs NOBC_V2'
    fig, axes = plot_ig_side_by_side([nobc_df, v2_df], ['NOBC', 'NOBC_V2'], title=title)
    fig.savefig(Q3_DIR / f'{basin}_{phys_type}_nobc_vs_nobcv2.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
compare_nobc_nobcv2('guerneville', 'NP')

In [ ]:
compare_nobc_nobcv2('guerneville', 'PHYS')

In [ ]:
compare_nobc_nobcv2('hopland', 'NP')

In [ ]:
compare_nobc_nobcv2('hopland', 'PHYS')

In [ ]:
compare_nobc_nobcv2('calpella', 'NP')

In [ ]:
compare_nobc_nobcv2('calpella', 'PHYS')